In [3]:
from scipy.stats import wilcoxon

# per-fold AUC and ACC values for H-optimus-0, CONCH, and UNI
h_AUC = ['0.9249', '0.9352', '0.9601', '0.8910', '0.8983']
h_ACC = ['0.8800', '0.9000', '0.9000', '0.8500', '0.8200']
c_AUC = ['0.8507', '0.8820', '0.9069', '0.8953', '0.8662']
c_ACC = ['0.8400', '0.7900', '0.8400', '0.8500', '0.8000']
u_AUC = ['0.9189', '0.9541', '0.9206', '0.9163', '0.9172']
u_ACC = ['0.8800', '0.8700', '0.8200', '0.8500', '0.8200']

# AUC comparisons
print("AUC:")
print("H-optimus-0 vs CONCH:", wilcoxon(h_AUC, c_AUC))
print("H-optimus-0 vs UNI:", wilcoxon(h_AUC, u_AUC))
print("CONCH vs UNI:", wilcoxon(c_AUC, u_AUC))

# ACC comparisons
print("\nACC:")
print("H-optimus-0 vs CONCH:", wilcoxon(h_ACC, c_ACC))
print("H-optimus-0 vs UNI:", wilcoxon(h_ACC, u_ACC))
print("CONCH vs UNI:", wilcoxon(c_ACC, u_ACC))


AUC:
H-optimus-0 vs CONCH: WilcoxonResult(statistic=np.float64(1.0), pvalue=np.float64(0.125))
H-optimus-0 vs UNI: WilcoxonResult(statistic=np.float64(6.0), pvalue=np.float64(0.8125))
CONCH vs UNI: WilcoxonResult(statistic=np.float64(0.0), pvalue=np.float64(0.0625))

ACC:
H-optimus-0 vs CONCH: WilcoxonResult(statistic=np.float64(0.0), pvalue=np.float64(0.125))
H-optimus-0 vs UNI: WilcoxonResult(statistic=np.float64(0.0), pvalue=np.float64(0.5))
CONCH vs UNI: WilcoxonResult(statistic=np.float64(2.0), pvalue=np.float64(0.375))


In [ ]:
# Multiple testing correction (Bonferroni)
print("\nBonferroni-corrected p-values:")
print("H-optimus-0 vs CONCH (AUC):", wilcoxon(h_AUC, c_AUC).pvalue * 3)
print("H-optimus-0 vs UNI (AUC):", wilcoxon(h_AUC, u_AUC).pvalue * 3)
print("CONCH vs UNI (AUC):", wilcoxon(c_AUC, u_AUC).pvalue * 3)
print("H-optimus-0 vs CONCH (ACC):", wilcoxon(h_ACC, c_ACC).pvalue * 3)
print("H-optimus-0 vs UNI (ACC):", wilcoxon(h_ACC, u_ACC).pvalue * 3)
print("CONCH vs UNI (ACC):", wilcoxon(c_ACC, u_ACC).pvalue * 3)


Bonferroni-corrected p-values:
H-optimus-0 vs CONCH (AUC): 0.375
H-optimus-0 vs UNI (AUC): 2.4375
CONCH vs UNI (AUC): 0.1875
H-optimus-0 vs CONCH (ACC): 0.375
H-optimus-0 vs UNI (ACC): 1.5
CONCH vs UNI (ACC): 1.125


In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from wsidata import open_wsi

# Set these from your server/session
slide_paths = []  # e.g., ['/path/to/slide1.mrxs', '/path/to/slide2.mrxs']
zarr_dir = "."  # e.g., '/path/to/zarr_cache'

# Feature norm distributions used by ABMIL sampling (see abmil.py: torch.linalg.norm(feats, dim=1))
feature_keys = {
    "H-optimus-0": "features_h-optimus-0",
    "CONCH": "features_conch",
    "UNI": "features_uni",
}

if len(slide_paths) == 0:
    raise ValueError("Please set `slide_paths` to your .mrxs files.")

rows = []
for i, slide_path in enumerate(slide_paths):
    zarr_path = os.path.join(zarr_dir, os.path.basename(slide_path).replace(".mrxs", ".zarr"))
    
    try:
        wsi = open_wsi(slide_path, zarr_path)
    except Exception:
        continue

    for model_name, feature_key in feature_keys.items():
        if feature_key not in wsi.tables:
            continue
        try:
            X = wsi.tables[feature_key].X
            if X is None or X.size == 0:
                continue
            norms = np.linalg.norm(X, axis=1)
            rows.extend(
                {
                    "model": model_name,
                    "feature_norm": float(v),
                    "slide": os.path.basename(slide_path),
                }
                for v in norms
            )
        except Exception:
            continue

norm_df = pd.DataFrame(rows)
if norm_df.empty:
    raise RuntimeError(
        "No feature norms found. Check `feature_keys`, `zarr_dir`, and whether tables exist in your zarr files."
    )

print(norm_df.groupby("model")["feature_norm"].agg(["count", "mean", "std", "median"]).round(4))

# --- Plot distributions ---
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.kdeplot(
    data=norm_df,
    x="feature_norm",
    hue="model",
    fill=True,
    common_norm=False,
    alpha=0.25,
    ax=axes[0],
)
axes[0].set_title("Tile Feature Norm Distribution")
axes[0].set_xlabel("L2 norm of tile embedding")

sns.boxplot(
    data=norm_df,
    x="model",
    y="feature_norm",
    ax=axes[1],
    showfliers=False,
    )
axes[1].set_title("Feature Norm by Model (boxplot)")
axes[1].set_xlabel("")
axes[1].set_ylabel("L2 norm")

# Histogram of feature norms
plt.figure(figsize=(7, 5))
sns.histplot(
    data=norm_df,
    x="feature_norm",
    hue="model",
    element="step",
    stat="density",
    common_norm=False,
    bins=50,
    )
plt.title("Feature Norm Histogram")
plt.xlabel("L2 norm")
plt.ylabel("Density")

plt.tight_layout()
plt.show()